# Clase 024 — Operaciones y alineación

**Parte 0** · VanderPlas cap. 3 § 3.4.

> 🎯 Alineación por index, apply vs vectorización, fill_value.

> ⏱️ ~60 min

## ⚙️ Setup

In [ ]:
import numpy as np
import pandas as pd
import time

## 1️⃣ Alineación automática

Operar dos Series alinea por **index**, no por posición:

In [ ]:
a = pd.Series([100, 200, 300], index=['x', 'y', 'z'])
b = pd.Series([10, 20, 30],    index=['y', 'z', 'w'])

print('a:'); print(a)
print('\nb:'); print(b)
print('\na + b — alinea, NaN donde no hay match:')
print(a + b)

## 2️⃣ `fill_value` evita propagar NaN

In [ ]:
print('a.add(b, fill_value=0):')
print(a.add(b, fill_value=0))
# x: 100+0=100  y: 200+10=210  z: 300+20=320  w: 0+30=30

## 3️⃣ `apply` — flexible pero lento por fila

**Regla**: si puedes hacerlo con ufunc/operadores vectorizados, **no uses apply**. Si necesitas lógica compleja por fila, sí.

In [ ]:
df = pd.DataFrame({
    'masa': [3750, 3800, 3250, 4400, 3700],
    'pico': [39.1, 39.5, 40.3, 36.7, 39.3],
})

# apply axis=1: una fila por iteración (lento)
def bmi_fila(row):
    return row['masa'] / (row['pico'] ** 2)

bmi_apply = df.apply(bmi_fila, axis=1)
print('con apply:')
print(bmi_apply.round(3))

# Vectorizado: una operación sobre todo el array (rápido)
bmi_vec = df['masa'] / (df['pico'] ** 2)
print('\nvectorizado:')
print(bmi_vec.round(3))
print(f'\niguales? {(bmi_apply.round(6) == bmi_vec.round(6)).all()}')

## 4️⃣ Benchmark apply vs vectorizado

In [ ]:
rng = np.random.default_rng(42)
grande = pd.DataFrame({
    'masa': rng.uniform(3000, 5000, 10_000),
    'pico': rng.uniform(35, 50, 10_000),
})

t0 = time.perf_counter(); grande.apply(bmi_fila, axis=1); t1 = time.perf_counter()
t2 = time.perf_counter(); grande['masa'] / (grande['pico'] ** 2); t3 = time.perf_counter()

print(f'apply        : {(t1-t0)*1000:.1f} ms')
print(f'vectorizado  : {(t3-t2)*1000:.2f} ms')
print(f'speedup      : {(t1-t0)/(t3-t2):.0f}×')

## 5️⃣ `map` para Series — recodificación con dict

Útil para mapear categorías a códigos o relabelar:

In [ ]:
species = pd.Series(['Adelie', 'Chinstrap', 'Gentoo', 'Adelie', 'Gentoo'])
codigo = species.map({'Adelie': 0, 'Chinstrap': 1, 'Gentoo': 2})
print(pd.DataFrame({'species': species, 'codigo': codigo}))

## 6️⃣ `df.map` — elementwise (era `applymap`)

Aplica una función a **cada celda** del DataFrame. Lento — úsalo solo cuando vectorización no aplica:

In [ ]:
df_num = pd.DataFrame({'A': [1.234, 5.678], 'B': [9.0, 0.1234]})
format_pct = df_num.map(lambda x: f'{x*100:.2f}%')
print(format_pct)

## 7️⃣ ufuncs NumPy preservan index

Pandas "sabe" NumPy — aplicar `np.log`, `np.sqrt`, etc., a una Series mantiene el index:

In [ ]:
s = pd.Series([1, 10, 100, 1000], index=['a', 'b', 'c', 'd'])
print('s:'); print(s)
print('\nnp.log(s):'); print(np.log(s).round(3))

## ✅ Checklist

- [ ] Sé que pandas alinea por index automáticamente
- [ ] Uso `fill_value` para evitar NaN en operaciones
- [ ] Prefiero vectorización a apply
- [ ] Uso `map` para recodificar Series con dict
- [ ] Sé que ufuncs NumPy preservan el index

## 📝 Homework

Ver `README.md`. BMI con apply vs vectorizado + benchmark, map species, alineación con fill_value.

## 🔗 Referencias

- VanderPlas cap. 3 § 3.4
- [pandas function application](https://pandas.pydata.org/docs/user_guide/basics.html#function-application)

➡️ **Siguiente:** [025 — Datos faltantes](../025-pandas-datos-faltantes/README.md)